# Branch check on the frozen-test cases

The frozen workflow stops at the first start that converges. The power-flow equations can have more than one solution,
and in two frozen-test cases GENCO's start and the classical starts reached different ones. This notebook reruns the 870
cases (rebuilt from their stored operating points and removed lines) with `branch_check=True`: every start is run, the
converged states are grouped into distinct solutions, and their decisions are compared.

It is reported separately from the frozen test. The first converged start must reproduce the frozen outcome of every
case, which also checks the newer module against the frozen run.

Runtime: T4 GPU, about 0.6 GB of downloads, 15 to 25 minutes. Output: `branch_check.zip`.

## Settings

In [ ]:
REPO_ID = "gridfm/genco-pf-contingency"
HF_REVISION = "5c625b00f5b79c8a7d4d27ecc133d4ecdf573af7"       # the revision every run used (unchanged since 1 October 2026)
GRAPHKIT_COMMIT = "95b90eecc7f5c2c86a9dd776f0c3f40434def22f"   # the commit every run used (branch head from 1 to 14 October 2026)
LOCAL_DIR = "genco-pf-contingency"
OUT_DIR = "branch_check"
GRAPHKIT_DIR = "gridfm-graphkit"
GRAPHKIT_BRANCH = "genco-paper-repro"
NETWORK = "Texas2k_case1_2016summerpeak"
MODEL_DIR = "model/training_bs_16"

LEVEL_BASE = 2
BASE_PARTS = [6, 16]                  # the N-2 data blocks of the frozen evaluation (its stored protocol)
N_CHECK = 16                          # base graphs on which the loaded model is checked
POLICY = {"tol_pu": 1e-8, "max_iter": 30, "order": ["genco", "base", "dc", "flat"], "line_limit": 1.0, "voltage_buses": "pq", "low_voltage_flag": 0.5,
          "recovery": {"max_iter": 100, "homotopy_steps": [0.25, 0.5, 0.75, 1.0]}, "screen_margin": 0.0454, "enforce_q_limits": False, "q_limit_rounds": 15}
BRANCH = {"branch_check": True, "branch_tol_pu": 1e-4, "branch_choice": "first"}     # the revised method; "first" keeps the frozen reference rule
KNOWN_CASES = [12742, 32934]          # the two cases found after the frozen run
REGION_HOPS, BETWEENNESS_SAMPLES = 4, 400
BATCH = 1
TOL, MAX_NR = 1e-8, 30
MAX_DOWNLOAD_GB = 3
SEED = 0
F1_CASES_CSV_B64 = "..."
F1_WORKFLOWS_CSV_B64 = "..."

In [ ]:
# [GPU] install and import

import importlib.util, os, subprocess, sys, time


def run(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-3000:])
        raise RuntimeError(f"Command failed: {cmd}")


import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
if not torch.cuda.is_available():
    print("WARNING: no GPU. Use Runtime > Change runtime type > T4 GPU, then Runtime > Run all again.\n"
          "The notebook also works on CPU, but the timing results would not be meaningful.")

if not os.path.isdir(GRAPHKIT_DIR):
    run(f"git clone -q -b {GRAPHKIT_BRANCH} https://github.com/gridfm/gridfm-graphkit.git {GRAPHKIT_DIR}")
    run(f"git -C {GRAPHKIT_DIR} checkout -q {GRAPHKIT_COMMIT}")   # pinned: the commit every run used
need = {"torch_geometric": "torch_geometric", "lightning": "lightning", "pytorch_lightning": "pytorch-lightning",
        "huggingface_hub": "huggingface_hub", "pyarrow": "pyarrow", "seaborn": "seaborn", "sklearn": "scikit-learn"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
if missing:
    run(f"{sys.executable} -m pip install -q " + " ".join(missing))

import torch_geometric   # imported before the replacement below, so PyTorch Geometric keeps its own native code paths
USED_SHIM = importlib.util.find_spec("torch_scatter") is None
if USED_SHIM:
    shim = os.path.abspath(os.path.join("shims", "torch_scatter"))
    os.makedirs(shim, exist_ok=True)
    with open(os.path.join(shim, "__init__.py"), "w") as f:
        f.write('"""Small pure-PyTorch stand-in for torch_scatter (only the functions gridfm-graphkit uses)."""\nimport torch\n\n__version__ = "0.0.0-shim"\n\n\ndef _prep(src, index, dim):\n    dim = dim if dim >= 0 else src.dim() + dim\n    if index.dim() != src.dim():\n        shape = [1] * src.dim()\n        shape[dim] = -1\n        index = index.view(shape).expand_as(src)\n    return index, dim\n\n\ndef _size(src, index, dim, dim_size):\n    size = list(src.shape)\n    if dim_size is None:\n        dim_size = int(index.max()) + 1 if index.numel() > 0 else 0\n    size[dim] = dim_size\n    return size\n\n\ndef scatter_add(src, index, dim=-1, out=None, dim_size=None):\n    index, dim = _prep(src, index, dim)\n    if out is None:\n        out = torch.zeros(_size(src, index, dim, dim_size), dtype=src.dtype, device=src.device)\n    return out.scatter_add_(dim, index, src)\n\n\nscatter_sum = scatter_add\n\n\ndef scatter_mean(src, index, dim=-1, out=None, dim_size=None):\n    total = scatter_add(src, index, dim, None, dim_size)\n    idx, d = _prep(src, index, dim)\n    count = torch.zeros_like(total).scatter_add_(d, idx, torch.ones_like(src)).clamp_(min=1)\n    return total / count\n\n\ndef scatter_max(src, index, dim=-1, out=None, dim_size=None):\n    idx, d = _prep(src, index, dim)\n    res = torch.full(_size(src, idx, d, dim_size), float("-inf"), dtype=src.dtype, device=src.device)\n    res = res.scatter_reduce(d, idx, src, reduce="amax", include_self=True)\n    res[torch.isinf(res)] = 0\n    return res, None\n\n\ndef scatter(src, index, dim=-1, out=None, dim_size=None, reduce="sum"):\n    if reduce in ("sum", "add"):\n        return scatter_add(src, index, dim, out, dim_size)\n    if reduce == "mean":\n        return scatter_mean(src, index, dim, out, dim_size)\n    if reduce == "max":\n        return scatter_max(src, index, dim, out, dim_size)[0]\n    raise NotImplementedError(reduce)\n')
    sys.path.insert(0, os.path.abspath("shims"))
    print("torch_scatter is not installed: using a small built-in replacement")
sys.path.insert(0, os.path.abspath(GRAPHKIT_DIR))

import gridfm_graphkit
from gridfm_graphkit.io.param_handler import NestedNamespace, load_normalizer, get_task_transforms, get_task
from gridfm_graphkit.training.callbacks import adapt_state_dict_for_model
from gridfm_graphkit.datasets.hetero_preprocess import build_hetero_data_for_scenario, merge_agg_gen_into_bus
from gridfm_graphkit.datasets.globals import VM_OUT, VA_OUT
from torch_geometric.loader import DataLoader
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch_geometric", torch_geometric.__version__, "| gridfm-graphkit from", os.path.abspath(GRAPHKIT_DIR), "| device:", DEVICE)

In [ ]:
# lightsim2grid: open-source C++ Newton-Raphson (KLU sparse solver), used here as the professional AC solver
import importlib.util, subprocess, sys
if importlib.util.find_spec("lightsim2grid") is None:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightsim2grid"], capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:])
print("lightsim2grid installed:", importlib.util.find_spec("lightsim2grid") is not None)

In [ ]:
import base64
GENCO_VERIFY_SOURCE = base64.b64decode('...').decode()
exec(GENCO_VERIFY_SOURCE)
print('genco_verify loaded (version 0.4.3: complex-voltage branch comparison, per-start decisions); lightsim2grid (KLU):', HAVE_KLU)

In [ ]:
# ===================== core functions =====================
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import scipy.sparse as sp
from scipy.sparse.linalg import splu

SN_MVA = 100.0
BUS_COLS = ["scenario", "bus", "Pd", "Qd", "Pg", "Qg", "Vm", "Va", "PQ", "PV", "REF",
            "min_vm_pu", "max_vm_pu", "GS", "BS", "Va_dc", "vn_kv"]
BR_COLS = ["scenario", "idx", "from_bus", "to_bus", "Yff_r", "Yff_i", "Yft_r", "Yft_i", "Ytf_r", "Ytf_i",
           "Ytt_r", "Ytt_i", "rate_a", "br_status", "pf_dc", "pt_dc"]
PRED_ALIASES = {"vm": ["vm_pu", "Vm_pred", "VM_pred", "vm_pred"], "va": ["va", "Va_pred", "VA_pred", "va_pred"]}


# ---------------------------------------------------------------- file helpers
def find_local(local_dir, k, table):
    base = Path(local_dir) / "data" / f"test_{k}"
    hits = sorted(base.rglob(f"{table}.parquet")) if base.exists() else []
    if not hits:
        raise FileNotFoundError(f"No {table}.parquet under {base}")
    return hits[0]


def find_pred(local_dir, k):
    base = Path(local_dir) / "mlflow" / "eval" / f"test_{k}"
    hits = sorted(base.rglob("*predictions.parquet")) if base.exists() else []
    hits = [h for h in hits if not re.search(r"_(gen|branch)_predictions\.parquet$", h.name)]
    if not hits:
        raise FileNotFoundError(f"No predictions.parquet under {base}")
    return hits[0]


def _dataset(path):
    return ds.dataset(str(path), format="parquet", partitioning="hive")


def read_cols(path, wanted, scen=None):
    dset = _dataset(path)
    cols = [c for c in wanted if c in dset.schema.names]
    flt = None
    if scen is not None:
        values = pa.array(np.asarray(scen, dtype=np.int64)).cast(dset.schema.field("scenario").type)
        flt = ds.field("scenario").isin(values)
    return dset.to_table(columns=cols, filter=flt).to_pandas()


def scenario_ids(path):
    return np.unique(_dataset(path).to_table(columns=["scenario"]).column("scenario").to_numpy().astype(np.int64))


# ---------------------------------------------------------------- one chunk of scenarios
def load_chunk(bus_path, br_path, pred_chunk, scen):
    """Arrays for a chunk of scenarios. pred_chunk: DataFrame with scenario, bus, vm, va (already filtered)."""
    bus = read_cols(bus_path, BUS_COLS, scen=scen)
    bus["scenario"] = bus["scenario"].astype(np.int64)
    bus = bus.sort_values(["scenario", "bus"], kind="stable").reset_index(drop=True)
    pred = pred_chunk.sort_values(["scenario", "bus"], kind="stable").reset_index(drop=True)
    S = len(scen)
    nb = len(bus) // S
    if len(bus) != S * nb or len(pred) != S * nb:
        raise RuntimeError(f"rows do not match: bus {len(bus)}, predictions {len(pred)}, {S} scenarios")
    ids = np.broadcast_to(np.arange(nb), (S, nb))
    if not (np.array_equal(bus["bus"].to_numpy().reshape(S, nb), ids) and np.array_equal(pred["bus"].to_numpy().reshape(S, nb), ids)):
        raise RuntimeError("bus numbers are not 0..nb-1 in every scenario")
    A = {c: bus[c].to_numpy(np.float64).reshape(S, nb) for c in BUS_COLS[2:] if c in bus.columns}
    A["PQ"], A["PV"], A["REF"] = A["PQ"] > 0.5, A["PV"] > 0.5, A["REF"] > 0.5
    A["Vm_g"] = pred["vm"].to_numpy(np.float64).reshape(S, nb)
    va = pred["va"].to_numpy(np.float64).reshape(S, nb)
    rad_err = np.nanmedian(np.abs(np.degrees(va) - A["Va"]))
    deg_err = np.nanmedian(np.abs(va - A["Va"]))
    A["va_unit"] = "rad" if rad_err <= deg_err else "deg"
    A["Va_g"] = np.degrees(va) if A["va_unit"] == "rad" else va
    br = read_cols(br_path, BR_COLS, scen=scen)
    A["sc"] = np.searchsorted(scen, br["scenario"].to_numpy().astype(np.int64))
    A["line"] = (br["idx"].to_numpy().astype(np.int64) if "idx" in br.columns
                 else br.groupby("scenario").cumcount().to_numpy().astype(np.int64))
    A["f"] = br["from_bus"].to_numpy().astype(np.int64)
    A["t"] = br["to_bus"].to_numpy().astype(np.int64)
    for y in ["Yff", "Yft", "Ytf", "Ytt"]:
        A[y] = br[y + "_r"].to_numpy(np.float64) + 1j * br[y + "_i"].to_numpy(np.float64)
    A["rate"] = br["rate_a"].to_numpy(np.float64)
    A["on"] = (br["br_status"].to_numpy(np.float64) > 0.5) if "br_status" in br.columns else (np.abs(A["Yff"]) > 0)
    pdc = br["pf_dc"].to_numpy(np.float64) if "pf_dc" in br.columns else np.full(len(br), np.nan)
    ptdc = br["pt_dc"].to_numpy(np.float64) if "pt_dc" in br.columns else pdc
    A["P_dc_abs"] = np.maximum(np.abs(pdc), np.abs(ptdc))
    A["S"], A["nb"], A["scen"] = S, nb, np.asarray(scen)
    return A


def branch_flows(A, V):
    sc, f, t = A["sc"], A["f"], A["t"]
    Vf, Vt = V[sc, f], V[sc, t]
    Sf = Vf * np.conj(A["Yff"] * Vf + A["Yft"] * Vt) * SN_MVA
    St = Vt * np.conj(A["Ytt"] * Vt + A["Ytf"] * Vf) * SN_MVA
    return Sf, St


def bus_mismatch(A, Vm, Sf, St):
    S, nb, sc, f, t = A["S"], A["nb"], A["sc"], A["f"], A["t"]
    add = lambda v, at: np.bincount(sc * nb + at, weights=v, minlength=S * nb).reshape(S, nb)
    P = (A["Pg"] - A["Pd"]) - (add(Sf.real, f) + add(St.real, t)) - A["GS"] * Vm**2 * SN_MVA
    Q = (A["Qg"] - A["Qd"]) - (add(Sf.imag, f) + add(St.imag, t)) + A["BS"] * Vm**2 * SN_MVA
    return P, Q


def _scen_max(values, sc, mask, S):
    out = np.full(S, np.nan)
    v = np.where(mask, values, -np.inf)
    tmp = np.full(S, -np.inf); np.maximum.at(tmp, sc, v)
    out[np.isfinite(tmp)] = tmp[np.isfinite(tmp)]
    return out


def chunk_tables(A, near_line=0.4, near_v=0.08):
    """Compact line-level and bus-level tables plus per-scenario signals for one chunk."""
    S, nb = A["S"], A["nb"]
    V_t = A["Vm"] * np.exp(1j * np.radians(A["Va"]))
    V_g = A["Vm_g"] * np.exp(1j * np.radians(A["Va_g"]))
    V_d = np.exp(1j * np.radians(A["Va_dc"]))
    Sf_t, St_t = branch_flows(A, V_t)
    Sf_g, St_g = branch_flows(A, V_g)
    Sf_d, St_d = branch_flows(A, V_d)
    P_t, _ = bus_mismatch(A, A["Vm"], Sf_t, St_t)
    P_g, Q_g = bus_mismatch(A, A["Vm_g"], Sf_g, St_g)
    resP = np.where(A["REF"], np.nan, np.abs(P_g))
    with np.errstate(all="ignore"):
        r_p99 = np.nanpercentile(resP, 99, axis=1)
    rated = A["on"] & (A["rate"] > 0)
    rate = np.where(rated, A["rate"], np.nan)
    ld = lambda Sf, St: np.maximum(np.abs(Sf), np.abs(St)) / rate
    ld_t, ld_g, ld_dS = ld(Sf_t, St_t), ld(Sf_g, St_g), ld(Sf_d, St_d)
    ld_dP = A["P_dc_abs"] / rate
    dc_failed = np.isnan(A["Va_dc"]).any(axis=1)
    keep = rated & ((ld_t >= near_line) | (ld_g >= near_line) | (np.nan_to_num(ld_dS) >= near_line) | (np.nan_to_num(ld_dP) >= near_line))
    lines = pd.DataFrame({
        "sidx": A["sc"][keep].astype(np.int32), "line": A["line"][keep].astype(np.int32),
        "ld_true": ld_t[keep].astype(np.float32), "ld_genco": ld_g[keep].astype(np.float32),
        "ld_dc_paper": ld_dS[keep].astype(np.float32), "ld_dc_p": ld_dP[keep].astype(np.float32),
    })
    lines.loc[dc_failed[lines.sidx.to_numpy()], ["ld_dc_paper", "ld_dc_p"]] = np.nan
    # PQ buses close to a voltage limit (truth or GENCO)
    PQ, vmin, vmax = A["PQ"], A["min_vm_pu"], A["max_vm_pu"]
    dist_t = np.minimum(A["Vm"] - vmin, vmax - A["Vm"])
    dist_g = np.minimum(A["Vm_g"] - vmin, vmax - A["Vm_g"])
    kb = PQ & ((dist_t <= near_v) | (dist_g <= near_v))
    si, bi = np.nonzero(kb)
    buses = pd.DataFrame({"sidx": si.astype(np.int32), "bus": bi.astype(np.int32),
                          "vm_true": A["Vm"][kb].astype(np.float32), "vm_genco": A["Vm_g"][kb].astype(np.float32),
                          "vmin": vmin[kb].astype(np.float32), "vmax": vmax[kb].astype(np.float32)})
    scen = pd.DataFrame({
        "scenario": A["scen"], "res_p99": r_p99,
        "n_lines_out": np.bincount(A["sc"][~A["on"]], minlength=S),
        "n_overload_true": np.bincount(A["sc"][rated], weights=(ld_t[rated] >= 1.0), minlength=S),
        "n_vviol_true": (PQ & ((A["Vm"] < vmin) | (A["Vm"] > vmax))).sum(axis=1),
        "err_vm_max": np.nanmax(np.where(PQ, np.abs(A["Vm_g"] - A["Vm"]), np.nan), axis=1),
        "worst_bus": np.nanargmax(np.where(PQ, np.abs(A["Vm_g"] - A["Vm"]), -1.0), axis=1),
        "err_load_max": _scen_max(np.abs(ld_g - ld_t), A["sc"], rated, S),
        "n_missed_overload_genco": np.bincount(A["sc"][rated], weights=((ld_t[rated] >= 1.0) & (ld_g[rated] < 1.0)), minlength=S),
        "n_missed_vviol_genco": (PQ & ((A["Vm"] < vmin) | (A["Vm"] > vmax)) & (A["Vm_g"] >= vmin) & (A["Vm_g"] <= vmax)).sum(axis=1),
        "dc_failed": dc_failed,
    })
    scen["worst_bus_kv"] = A["vn_kv"][np.arange(S), scen["worst_bus"].to_numpy()] if "vn_kv" in A else np.nan
    checks = {"truth_mismatch_max_MW": float(np.nanmax(np.abs(np.where(A["REF"], 0, P_t)))),
              "va_unit": A["va_unit"]}
    return lines, buses, scen, checks


# ---------------------------------------------------------------- margins
def fit_exponent(e, r, n_bins=8, q=0.99, min_count=200):
    """Slope of log(quantile of error) against log(scenario residual), from bins of the residual. Clipped to [0, 2]."""
    e, r = np.asarray(e, float), np.asarray(r, float)
    ok = np.isfinite(e) & np.isfinite(r) & (r > 0)
    e, r = e[ok], r[ok]
    if len(e) < 3 * min_count:
        return 0.0, []
    if len(e) < n_bins * min_count:
        n_bins = max(3, len(e) // min_count)
    edges = np.unique(np.quantile(r, np.linspace(0, 1, n_bins + 1)))
    idx = np.clip(np.searchsorted(edges, r, side="right") - 1, 0, len(edges) - 2)
    xs, ys = [], []
    for b in range(len(edges) - 1):
        m = idx == b
        if m.sum() >= min_count:
            qq = np.quantile(e[m], q)
            if qq > 0:
                xs.append(np.log(np.median(r[m]))); ys.append(np.log(qq))
    if len(xs) < 3:
        return 0.0, list(zip(xs, ys))
    return float(np.clip(np.polyfit(xs, ys, 1)[0], 0.0, 2.0)), list(zip(xs, ys))


def scale_of(r, r_ref, b):
    return np.power(np.maximum(np.asarray(r, float), 1e-9) / r_ref, b)


# ---------------------------------------------------------------- screening curves
def screening_curve(score, positive, sidx_global, n_scen, thresholds):
    """Flag an item if score >= threshold. Returns recall, false alarms per scenario, share of scenarios with a flag."""
    score = np.asarray(score, float); positive = np.asarray(positive, bool)
    ok = np.isfinite(score)
    score, positive, sidx = score[ok], positive[ok], np.asarray(sidx_global)[ok]
    n_pos = max(int(positive.sum()), 1)
    order = np.argsort(-score, kind="stable")
    s_desc = -score[order]                          # ascending
    cum_pos = np.r_[0, np.cumsum(positive[order])]
    cum_neg = np.r_[0, np.cumsum(~positive[order])]
    smax = np.full(n_scen, -np.inf); np.maximum.at(smax, sidx, score)
    smax_sorted = np.sort(smax)
    th = np.asarray(thresholds, float)
    n = np.searchsorted(s_desc, -th, side="right")   # items with score >= th
    tp, fp = cum_pos[n], cum_neg[n]
    sent = n_scen - np.searchsorted(smax_sorted, th, side="left")
    return pd.DataFrame({"threshold": th, "recall": tp / n_pos, "misses": n_pos - tp,
                         "false_alarms_per_scenario": fp / n_scen, "share_scenarios_flagged": sent / n_scen})


# ---------------------------------------------------------------- Newton-Raphson (polar, MATPOWER formulation)
def build_ybus(nb, f, t, Yff, Yft, Ytf, Ytt, GS, BS):
    rows = np.concatenate([f, f, t, t, np.arange(nb)])
    cols = np.concatenate([f, t, f, t, np.arange(nb)])
    vals = np.concatenate([Yff, Yft, Ytf, Ytt, GS + 1j * BS])
    return sp.csr_matrix((vals, (rows, cols)), shape=(nb, nb))


def dsbus_dv(Ybus, V):
    Ibus = Ybus @ V
    dV, dI, dVn = sp.diags(V), sp.diags(Ibus), sp.diags(V / np.abs(V))
    dS_dVm = dV @ (Ybus @ dVn).conj() + dI.conj() @ dVn
    dS_dVa = 1j * dV @ (dI - Ybus @ dV).conj()
    return dS_dVm.tocsr(), dS_dVa.tocsr()


def newton(Ybus, Sspec, V0, pv, pq, tol=1e-8, max_it=20, n_steps=None):
    """Returns final V, iterations done, list of max mismatch (p.u.) after each iteration, first-step correction size."""
    V = V0.astype(complex).copy()
    Va, Vm = np.angle(V), np.abs(V)
    pvpq = np.r_[pv, pq]
    n1 = len(pvpq)

    def mis(V):
        S = V * np.conj(Ybus @ V) - Sspec
        return np.r_[S[pvpq].real, S[pq].imag]

    F = mis(V); norms = [float(np.max(np.abs(F)))]; it = 0; first_step = np.nan
    while True:
        if n_steps is None and (norms[-1] <= tol or it >= max_it):
            break
        if n_steps is not None and it >= n_steps:
            break
        dS_dVm, dS_dVa = dsbus_dv(Ybus, V)
        J = sp.vstack([sp.hstack([dS_dVa[pvpq][:, pvpq].real, dS_dVm[pvpq][:, pq].real]),
                       sp.hstack([dS_dVa[pq][:, pvpq].imag, dS_dVm[pq][:, pq].imag])], format="csc")
        try:
            dx = -splu(J).solve(F)
        except RuntimeError:
            norms.append(np.inf); break
        if it == 0:
            first_step = float(np.max(np.abs(dx)))
        Va[pvpq] += dx[:n1]; Vm[pq] += dx[n1:]
        V = Vm * np.exp(1j * Va)
        F = mis(V); it += 1; norms.append(float(np.max(np.abs(F))))
        if not np.isfinite(norms[-1]) or norms[-1] > 1e6:
            break
    return V, it, norms, first_step


def scenario_system(A, i):
    """Ybus, specified power and bus sets for scenario i of a chunk."""
    nb = A["nb"]
    m = (A["sc"] == i) & A["on"]
    Y = build_ybus(nb, A["f"][m], A["t"][m], A["Yff"][m], A["Yft"][m], A["Ytf"][m], A["Ytt"][m], A["GS"][i], A["BS"][i])
    Sspec = (A["Pg"][i] - A["Pd"][i] + 1j * (A["Qg"][i] - A["Qd"][i])) / SN_MVA
    pv, pq = np.nonzero(A["PV"][i])[0], np.nonzero(A["PQ"][i])[0]
    return Y, Sspec, pv, pq


def extract_scenario(A, i):
    """Keep only what Newton needs for scenario i (small, so it can be stored)."""
    m = A["sc"] == i
    out = {key: A[key][i].copy() for key in ["Pd", "Qd", "Pg", "Qg", "Vm", "Va", "PQ", "PV", "REF", "GS", "BS",
                                             "Va_dc", "Vm_g", "Va_g", "min_vm_pu", "max_vm_pu", "vn_kv"] if key in A}
    for key in ["f", "t", "Yff", "Yft", "Ytf", "Ytt", "rate", "on", "line"]:
        if key in A:
            out[key] = A[key][m].copy()
    out["nb"] = A["nb"]; out["scenario"] = int(A["scen"][i])
    return out


def run_newton_cases(d, tol=1e-8, max_it=20):
    """Flat start, DC start, GENCO start (to convergence) and GENCO + exactly one step, for one stored scenario."""
    nb = d["nb"]
    on = d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    Sspec = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    pv, pq, ref = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0], np.nonzero(d["REF"])[0]
    Vm_set = np.where(d["PQ"], 1.0, d["Vm"])          # setpoints at PV and slack, 1.0 elsewhere
    Va_ref = np.where(d["REF"], d["Va"], 0.0)
    V_true = d["Vm"] * np.exp(1j * np.radians(d["Va"]))

    def start(Vm, Va_deg):
        Vm = np.where(d["PQ"], Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    starts = {"flat": start(Vm_set, Va_ref), "dc": start(np.ones(nb), np.nan_to_num(d["Va_dc"])),
              "genco": start(d["Vm_g"], d["Va_g"])}
    res = {"scenario": d["scenario"]}
    for name, V0 in starts.items():
        t0 = time.perf_counter()
        V, it, norms, step1 = newton(Y, Sspec, V0, pv, pq, tol=tol, max_it=max_it)
        res[f"seconds_{name}"] = time.perf_counter() - t0
        conv = norms[-1] <= tol
        res[f"iters_{name}"] = it if conv else np.nan
        res[f"converged_{name}"] = bool(conv)
        res[f"start_mismatch_{name}"] = norms[0]
        if conv:
            res[f"max_dV_vs_truth_{name}"] = float(np.max(np.abs(V - V_true)))
        if name == "genco":
            res["first_step_size"] = step1
    t0 = time.perf_counter()
    V1, _, norms1, _ = newton(Y, Sspec, starts["genco"], pv, pq, n_steps=1)
    res["seconds_genco_1step"] = time.perf_counter() - t0
    res["mismatch_after_1_step"] = norms1[-1]
    rated = on & (d["rate"] > 0)

    def loading(V):
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[rated] / d["rate"][rated]

    ld_t = loading(V_true)
    over_t = ld_t >= 1.0
    vio_t = d["PQ"] & ((d["Vm"] < d["min_vm_pu"]) | (d["Vm"] > d["max_vm_pu"]))
    for name, V in [("genco", starts["genco"]), ("genco_1step", V1)]:
        ld = loading(V); Vm = np.abs(V)
        vio = d["PQ"] & ((Vm < d["min_vm_pu"]) | (Vm > d["max_vm_pu"]))
        res[f"err_vm_max_{name}"] = float(np.max(np.abs(Vm - d["Vm"])[d["PQ"]])) if d["PQ"].any() else np.nan
        res[f"err_load_max_{name}"] = float(np.max(np.abs(ld - ld_t))) if len(ld) else np.nan
        res[f"missed_overloads_{name}"] = int((over_t & (ld < 1.0)).sum())
        res[f"missed_vviol_{name}"] = int((vio_t & ~vio).sum())
    res["true_overloads"] = int(over_t.sum()); res["true_vviol"] = int(vio_t.sum())
    return res


def newton_trace(Ybus, Sspec, V0, pv, pq, tol=1e-8, max_it=10):
    """List of (V, max mismatch p.u., seconds for this step) after 0, 1, 2, ... Newton steps."""
    V = V0.astype(complex).copy()
    Va, Vm = np.angle(V), np.abs(V)
    pvpq = np.r_[pv, pq]; n1 = len(pvpq)

    def mis(V):
        S = V * np.conj(Ybus @ V) - Sspec
        return np.r_[S[pvpq].real, S[pq].imag]

    F = mis(V)
    out = [(V.copy(), float(np.max(np.abs(F))), 0.0)]
    for _ in range(max_it):
        if out[-1][1] <= tol:
            break
        t0 = time.perf_counter()
        dS_dVm, dS_dVa = dsbus_dv(Ybus, V)
        J = sp.vstack([sp.hstack([dS_dVa[pvpq][:, pvpq].real, dS_dVm[pvpq][:, pq].real]),
                       sp.hstack([dS_dVa[pq][:, pvpq].imag, dS_dVm[pq][:, pq].imag])], format="csc")
        try:
            dx = -splu(J).solve(F)
        except RuntimeError:
            break
        Va[pvpq] += dx[:n1]; Vm[pq] += dx[n1:]
        V = Vm * np.exp(1j * Va)
        F = mis(V)
        out.append((V.copy(), float(np.max(np.abs(F))), time.perf_counter() - t0))
        if not np.isfinite(out[-1][1]) or out[-1][1] > 1e6:
            break
    return out


def run_trace_cases(d, tol=1e-8, max_it=10, line_margin=0.0, v_margin=0.0):
    """Decision metrics after every Newton step from flat, DC and GENCO starts, for one stored scenario.
    Columns ending in _m use the safety margins (flag a line from 1 - line_margin, a bus within v_margin of a limit)."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    Sspec = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    pv, pq = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]
    PQ, lo, hi = d["PQ"], d["min_vm_pu"], d["max_vm_pu"]
    rated = on & (d["rate"] > 0)
    V_true = d["Vm"] * np.exp(1j * np.radians(d["Va"]))

    def loading(V):
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[rated] / d["rate"][rated]

    ld_t = loading(V_true)
    over_t = ld_t >= 1.0
    vio_t = PQ & ((d["Vm"] < lo) | (d["Vm"] > hi))

    def start(Vm, Va_deg):
        Vm = np.where(PQ, Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    starts = {"flat": start(np.ones(nb), np.zeros(nb)), "dc": start(np.ones(nb), np.nan_to_num(d["Va_dc"])),
              "genco": start(d["Vm_g"], d["Va_g"])}
    rows = []
    for name, V0 in starts.items():
        trace = newton_trace(Y, Sspec, V0, pv, pq, tol=tol, max_it=max_it)
        converged = trace[-1][1] <= tol
        for j, (V, mism, secs) in enumerate(trace):
            ld, Vm = loading(V), np.abs(V)
            vio = PQ & ((Vm < lo) | (Vm > hi))
            vio_m = PQ & ((Vm < lo + v_margin) | (Vm > hi - v_margin))
            fl_m = ld >= 1.0 - line_margin
            rows.append({"scenario": d["scenario"], "start": name, "steps": j, "mismatch": mism, "seconds_step": secs,
                         "converged_at_end": converged, "iterations_to_converge": len(trace) - 1 if converged else np.nan,
                         "err_vm_max": float(np.max(np.abs(Vm - d["Vm"])[PQ])) if PQ.any() else np.nan,
                         "err_load_max": float(np.max(np.abs(ld - ld_t))) if len(ld) else np.nan,
                         "missed_overloads": int((over_t & (ld < 1.0)).sum()), "false_overloads": int((~over_t & (ld >= 1.0)).sum()),
                         "missed_vviol": int((vio_t & ~vio).sum()), "false_vviol": int((~vio_t & vio).sum()),
                         "missed_overloads_m": int((over_t & ~fl_m).sum()), "false_overloads_m": int((~over_t & fl_m).sum()),
                         "missed_vviol_m": int((vio_t & ~vio_m).sum()), "false_vviol_m": int((~vio_t & vio_m).sum()),
                         "true_overloads": int(over_t.sum()), "true_vviol": int(vio_t.sum())})
    return rows


# ===================== GENCO helpers =====================
from scipy.sparse.csgraph import dijkstra


class ScenarioEval:
    """Truth-based checks of any voltage state for one stored scenario (from extract_scenario)."""

    def __init__(self, d):
        self.d = d
        nb, on = d["nb"], d["on"]
        self.Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
        self.Sspec = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
        self.pv, self.pq = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]
        self.nonref = np.nonzero(~d["REF"])[0]
        self.rated = on & (d["rate"] > 0)
        Vt = d["Vm"] * np.exp(1j * np.radians(d["Va"]))
        self.ld_t = self.loading(Vt)
        self.over_t = self.ld_t >= 1.0
        self.vio_t = d["PQ"] & ((d["Vm"] < d["min_vm_pu"]) | (d["Vm"] > d["max_vm_pu"]))

    def loading(self, V):
        d = self.d
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[self.rated] / d["rate"][self.rated]

    def state(self, Vm, Va_deg):
        d = self.d
        Vm = np.where(d["PQ"], Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    def metrics(self, Vm, Va_deg):
        d = self.d
        V = self.state(Vm, Va_deg)
        S = V * np.conj(self.Y @ V) - self.Sspec
        Pm, Qm = np.abs(S.real), np.abs(S.imag)
        ld = self.loading(V)
        Vabs = np.abs(V)
        vio = d["PQ"] & ((Vabs < d["min_vm_pu"]) | (Vabs > d["max_vm_pu"]))
        err = np.abs(Vabs - d["Vm"])
        return {"err_vm_max": float(err[d["PQ"]].max()) if d["PQ"].any() else np.nan,
                "err_load_max": float(np.max(np.abs(ld - self.ld_t))) if len(ld) else np.nan,
                "mismatch_pu": float(max(Pm[np.r_[self.pv, self.pq]].max(), Qm[self.pq].max())),
                "res_p99_MW": float(np.percentile(Pm[self.nonref], 99) * SN_MVA),
                "missed_overloads": int((self.over_t & (ld < 1.0)).sum()),
                "missed_vviol": int((self.vio_t & ~vio).sum())}

    def bus_errors(self, Vm):
        return np.where(self.d["PQ"], np.abs(Vm - self.d["Vm"]), np.nan)


def thevenin_pq(d, block=200):
    """|Z_ii| for every PQ bus with PV and slack buses held at fixed voltage: how much the voltage at a bus
    moves per unit of current injected there (large = electrically weak spot)."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    pq = np.nonzero(d["PQ"])[0]
    lu = splu(Y[pq][:, pq].tocsc())
    n = len(pq); z = np.empty(n, complex)
    for s in range(0, n, block):
        idx = np.arange(s, min(n, s + block))
        E = np.zeros((n, len(idx)), complex); E[idx, np.arange(len(idx))] = 1.0
        X = lu.solve(E)
        z[idx] = X[idx, np.arange(len(idx))]
    return pq, np.abs(z)


def hops_to_voltage_control(d):
    """Number of in-service lines between each bus and the nearest PV or slack bus."""
    nb, on = d["nb"], d["on"]
    A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
    src = np.nonzero(d["PV"] | d["REF"])[0]
    dist = dijkstra(A, directed=False, unweighted=True, indices=src, min_only=True)
    return dist


def dc_solve_seconds(d, reps=3):
    """Time of one DC power flow (one sparse solve on the susceptance matrix), for the cost comparison."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    keep = np.nonzero(~d["REF"])[0]
    P = ((d["Pg"] - d["Pd"]) / SN_MVA)[keep]
    t0 = time.perf_counter()
    for _ in range(reps):
        B = (-Y.imag)[keep][:, keep].tocsc()
        splu(B).solve(P)
    return (time.perf_counter() - t0) / reps


# ===================== outage construction =====================
from scipy.sparse.csgraph import connected_components
try:
    from lightsim2grid.algorithm import NR_KLU as _KLU
    HAVE_KLU = True
except Exception:
    try:
        from lightsim2grid.solver import KLUSolver as _KLU
        HAVE_KLU = True
    except Exception:
        HAVE_KLU = False


def system_of(d):
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    S = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    return Y, S, np.nonzero(d["REF"])[0], np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]


def klu_solve(Y, S, V0, ref, pv, pq, max_iter, tol=1e-8):
    # lightsim2grid's C++ Newton-Raphson with the KLU sparse solver. Returns V, converged, iterations, seconds.
    w = np.zeros(Y.shape[0]); w[ref] = 1.0 / len(ref)
    Yc = Y.tocsc()
    s = _KLU()
    t0 = time.perf_counter()
    ok = s.solve(Yc, np.asarray(V0, complex), np.asarray(S, complex), ref.astype(np.int32), w,
                 pv.astype(np.int32), pq.astype(np.int32), int(max_iter), float(tol))
    dt = time.perf_counter() - t0
    V = np.array(s.get_V(), dtype=complex, copy=True)
    return V, bool(ok), int(s.get_nb_iter()), dt


def dc_angles(d):
    # DC power flow on the current topology (series susceptances, no shunts, no phase shifts). Returns angles in degrees.
    nb, on = d["nb"], d["on"]
    f, t, b = d["f"][on], d["t"][on], d["Yft"][on].imag
    B = sp.csr_matrix((np.r_[b, b, -b, -b], (np.r_[f, t, f, t], np.r_[f, t, t, f])), shape=(nb, nb)).tocsc()
    ref = np.nonzero(d["REF"])[0]
    keep = np.setdiff1d(np.arange(nb), ref)
    P = ((d["Pg"] - d["Pd"]) / SN_MVA)[keep]
    th = np.zeros(nb)
    th[keep] = splu(B[keep][:, keep]).solve(P)
    return np.degrees(th) + float(d["Va"][ref[0]])


def connected(nb, f, t):
    A = sp.csr_matrix((np.ones(len(f)), (f, t)), shape=(nb, nb))
    return connected_components(A, directed=False)[0] == 1


def pick_outages(d, extra, mode, rng):
    # Indices (into the scenario's branch rows) of `extra` more lines to remove, keeping the grid in one piece.
    # mode "clustered": the lines nearest (in hops) to a random centre bus, like a storm footprint.
    # mode "uniform": random lines anywhere, like the released test sets.
    nb = d["nb"]; on = d["on"].copy()
    cand = np.nonzero(on & (d["rate"] > 0))[0]
    hop = None
    if mode == "clustered":
        center = int(rng.integers(nb))
        A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
        hop = dijkstra(A, directed=False, unweighted=True, indices=center)
        dist = np.minimum(hop[d["f"][cand]], hop[d["t"][cand]])
        order = cand[np.lexsort((rng.random(len(cand)), dist))]
    else:
        order = rng.permutation(cand)
    removed = []
    for i in order:
        if len(removed) == extra:
            break
        trial = on.copy(); trial[i] = False
        if connected(nb, d["f"][trial], d["t"][trial]):
            on = trial; removed.append(int(i))
    if len(removed) < extra:
        return None, None
    radius = float(np.max(np.minimum(hop[d["f"][removed]], hop[d["t"][removed]]))) if hop is not None else np.nan
    return np.array(removed), radius


def solve_new_topology(d, removed, tol=1e-8, max_iter=30):
    # Exact AC solution after removing `removed`; starts: the base solution, then flat, then DC.
    d2 = {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in d.items()}
    d2["on"][removed] = False
    Y, S, ref, pv, pq = system_of(d2)
    base = d["Vm"] * np.exp(1j * np.radians(d["Va"]))
    flat = np.where(d2["PQ"], 1.0, d2["Vm"]) * np.exp(1j * np.radians(np.where(d2["REF"], d2["Va"], 0.0)))
    d2["Va_dc"] = dc_angles(d2)
    dc = np.where(d2["PQ"], 1.0, d2["Vm"]) * np.exp(1j * np.radians(d2["Va_dc"]))
    for name, V0 in [("base", base), ("flat", flat), ("dc", dc)]:
        if HAVE_KLU:
            V, ok, it, _ = klu_solve(Y, S, V0, ref, pv, pq, max_iter, tol)
        else:
            V, it, norms, _ = newton(Y, S, V0, pv, pq, tol=tol, max_it=max_iter)
            ok = norms[-1] <= tol
        if ok:
            d2["Vm"], d2["Va"] = np.abs(V), np.degrees(np.angle(V))
            d2["solved_from"] = name
            return d2
    return None




"""J1 core: true AC power-flow voltage sensitivities from the Jacobian, a finite-difference check of them,
hop distances in the grid graph, and the comparison metrics. Uses build_ybus / dsbus_dv / newton / SN_MVA from
the step-1/6 core."""
import numpy as np
import scipy.sparse as sp
from scipy.sparse.csgraph import shortest_path
from scipy.sparse.linalg import splu


def ac_system(d):
    """Ybus, true complex voltages, specified injections (p.u.) and bus sets of one stored scenario."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    V = d["Vm"] * np.exp(1j * np.radians(d["Va"]))
    S = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    pv, pq = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]
    return Y, V, S, pv, pq


def power_flow_jacobian(Y, V, pv, pq):
    """Jacobian of [P mismatch at pv+pq; Q mismatch at pq] with respect to [angle at pv+pq; magnitude at pq]."""
    pvpq = np.r_[pv, pq]
    dS_dVm, dS_dVa = dsbus_dv(Y, V)
    return sp.vstack([sp.hstack([dS_dVa[pvpq][:, pvpq].real, dS_dVm[pvpq][:, pq].real]),
                      sp.hstack([dS_dVa[pq][:, pvpq].imag, dS_dVm[pq][:, pq].imag])], format="csc"), pvpq


def true_sensitivity_rows(d, targets):
    """For each target bus (PQ in this scenario): dVm_target / dPd_j and dVm_target / dQd_j for every bus j,
    in p.u. voltage per p.u. load (100 MW or 100 Mvar). A load increase is a lower injection, hence the minus sign.
    Loads at the slack bus (P) and at voltage-controlled buses (Q) do not move any voltage: those entries are 0."""
    Y, V, S, pv, pq = ac_system(d)
    J, pvpq = power_flow_jacobian(Y, V, pv, pq)
    lu = splu(J.T.tocsc())
    n1, nb = len(pvpq), d["nb"]
    pos_pq = {int(b): n1 + i for i, b in enumerate(pq)}
    out = {}
    for t in targets:
        if int(t) not in pos_pq:
            continue
        e = np.zeros(J.shape[0]); e[pos_pq[int(t)]] = 1.0
        y = lu.solve(e)
        TP, TQ = np.zeros(nb), np.zeros(nb)
        TP[pvpq] = -y[:n1]
        TQ[pq] = -y[n1:]
        out[int(t)] = (TP, TQ)
    return out


def ac_fd_check(d, t, delta_pu=0.01):
    """Finite-difference check of the Jacobian sensitivities. Converge the base case, raise the reactive load at
    bus t by delta_pu (1 Mvar for 0.01), converge again, and compare dVm_t / dQd_t from the two solutions with the
    Jacobian value at the base solution. Returns (finite difference, Jacobian)."""
    Y, V, S, pv, pq = ac_system(d)
    V0, _, n0, _ = newton(Y, S, V, pv, pq, tol=1e-11, max_it=20)
    S2 = S.copy(); S2[t] -= 1j * delta_pu
    V2, _, n2, _ = newton(Y, S2, V0, pv, pq, tol=1e-11, max_it=20)
    if not n0 or not n2 or n0[-1] > 1e-9 or n2[-1] > 1e-9:
        return np.nan, np.nan
    J, pvpq = power_flow_jacobian(Y, V0, pv, pq)
    row = np.nonzero(pq == t)[0]
    e = np.zeros(J.shape[0]); e[len(pvpq) + int(row[0])] = 1.0
    jac = float(-splu(J.T.tocsc()).solve(e)[len(pvpq) + int(row[0])])
    return float((np.abs(V2[t]) - np.abs(V0[t])) / delta_pu), jac


def hop_distances(d, targets):
    """Number of in-service lines between each target and every bus (inf if not connected)."""
    nb, on = d["nb"], d["on"]
    f, t = d["f"][on], d["t"][on]
    A = sp.csr_matrix((np.ones(2 * len(f)), (np.r_[f, t], np.r_[t, f])), shape=(nb, nb))
    return shortest_path(A, unweighted=True, directed=False, indices=np.asarray(targets, dtype=np.int64))


def row_metrics(gP, gQ, tP, tQ, pq_mask, pv_mask):
    """Compare GENCO's sensitivity row with the true one (same units). Only loads that can move voltages enter the
    gain and cosine: P at every non-slack bus, Q at PQ buses. Leakage: share of GENCO's reactive-load sensitivity that
    sits on voltage-controlled buses, where the true value is exactly 0."""
    g = np.r_[gP[pq_mask | pv_mask], gQ[pq_mask]]
    tr = np.r_[tP[pq_mask | pv_mask], tQ[pq_mask]]
    tt = float(tr @ tr)
    gain = float(g @ tr / tt) if tt > 0 else np.nan
    cos = float(g @ tr / (np.linalg.norm(g) * np.sqrt(tt))) if tt > 0 and np.linalg.norm(g) > 0 else np.nan
    rel_err = float(np.linalg.norm(g - tr) / np.sqrt(tt)) if tt > 0 else np.nan
    leak = float(np.abs(gQ[~pq_mask]).sum() / max(np.abs(gQ).sum(), 1e-30))
    gq, tq = gQ[pq_mask], tQ[pq_mask]
    tqq = float(tq @ tq)
    gain_q = float(gq @ tq / tqq) if tqq > 0 else np.nan
    cos_q = float(gq @ tq / (np.linalg.norm(gq) * np.sqrt(tqq))) if tqq > 0 and np.linalg.norm(gq) > 0 else np.nan
    return {"gain": gain, "cosine": cos, "relative_error": rel_err, "gain_Q": gain_q, "cosine_Q": cos_q, "pv_q_leakage": leak}


def hop_profile(abs_row, hops, max_hop=8):
    """Sum of |sensitivity| at each hop distance 0..max_hop (the last bin holds max_hop and beyond)."""
    h = np.where(np.isfinite(hops), np.minimum(hops, max_hop), max_hop).astype(int)
    return np.bincount(h, weights=abs_row, minlength=max_hop + 1)[: max_hop + 1]


def slope_through(x, y):
    """Least-squares slope of y on x (both centred): how strongly GENCO's voltage follows the true one."""
    x = np.asarray(x, float) - np.mean(x); y = np.asarray(y, float) - np.mean(y)
    return float(x @ y / (x @ x)) if x @ x > 0 else np.nan


"""K1 core: one storm topology with every load and every non-slack generator scaled by a load factor lambda (the
slack takes the rest), the AC truth along lambda up to the point where the AC equations stop having a solution (the
"nose"), a signature that the nose is genuine, and the screening decisions GENCO, DC and the Newton check make at
every lambda. Uses build_ybus / dsbus_dv / newton / newton_trace / klu_solve / system_of / dc_angles / SN_MVA /
HAVE_KLU from the step-1/6 core and power_flow_jacobian from the J1 core."""
import time

import numpy as np
import scipy.sparse as sp
from scipy.sparse.csgraph import shortest_path
from scipy.sparse.linalg import splu


class LambdaCase:
    """One topology; injections scaled by lambda: P at every non-slack bus = lambda * (Pg - Pd), Q at PQ buses =
    Qg - lambda * Qd (constant power factor), generator voltage set-points unchanged, the slack takes the rest."""

    def __init__(self, d):
        self.d = d
        self.Y, _, self.ref, self.pv, self.pq = system_of(d)
        self.nb = d["nb"]
        self.PQ, self.PV, self.REF = d["PQ"], d["PV"], d["REF"]
        self.nonref = np.nonzero(~d["REF"])[0]
        self.P0 = (d["Pg"] - d["Pd"]) / SN_MVA
        self.Qg, self.Qd = d["Qg"] / SN_MVA, d["Qd"] / SN_MVA
        self.rated = d["on"] & (d["rate"] > 0)
        self.rate = d["rate"][self.rated]
        self.vmin, self.vmax = d["min_vm_pu"], d["max_vm_pu"]
        Vm_set = np.where(d["PQ"], 1.0, d["Vm"])
        self.flat = Vm_set * np.exp(1j * np.radians(np.where(d["REF"], d["Va"], 0.0)))
        self.dc = Vm_set * np.exp(1j * np.radians(np.nan_to_num(d["Va_dc"])))
        self.ld_dc1 = self.loading(np.exp(1j * np.radians(np.nan_to_num(d["Va_dc"]))))   # DC state, AC branch equations

    # ---------------------------------------------------------------- physics at a given lambda
    def S(self, lam):
        return lam * self.P0 + 1j * (self.Qg - lam * self.Qd)

    def state(self, Vm, Va_deg):
        d = self.d
        Vm = np.where(d["PQ"], Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    def solve(self, lam, V0, max_iter=30, tol=1e-8):
        """Newton-Raphson from V0 at this lambda. Returns V, converged, iterations, seconds."""
        S = self.S(lam)
        if HAVE_KLU:
            return klu_solve(self.Y, S, V0, self.ref, self.pv, self.pq, max_iter, tol)
        t0 = time.perf_counter()
        V, it, norms, _ = newton(self.Y, S, V0, self.pv, self.pq, tol=tol, max_it=max_iter)
        return V, bool(norms[-1] <= tol), it, time.perf_counter() - t0

    def solve_any(self, lam, starts, max_iter=30, tol=1e-8):
        """First converged solution over a list of (name, V0) starts; (None, None) if none converges."""
        for name, V0 in starts:
            V, ok, _, _ = self.solve(lam, V0, max_iter, tol)
            if ok and np.all(np.isfinite(V)) and np.abs(V).min() > 0.2:
                return V, name
        return None, None

    def loading(self, V):
        d = self.d
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[self.rated] / self.rate

    def dc_loading(self, lam):
        """DC power flow at this lambda: DC angles scale with lambda, flows through the AC branch equations."""
        return self.loading(np.exp(1j * np.radians(lam * np.nan_to_num(self.d["Va_dc"]))))

    def mismatch(self, lam, V):
        """Active mismatch (MW) at non-slack buses and reactive mismatch (Mvar) at PQ buses of a voltage state."""
        S = (V * np.conj(self.Y @ V) - self.S(lam)) * SN_MVA
        return np.abs(S.real[self.nonref]), np.abs(S.imag[self.pq])

    def jacobian(self, V):
        return power_flow_jacobian(self.Y, V, self.pv, self.pq)[0]

    # ---------------------------------------------------------------- what a screen would say about a state
    def decisions(self, lam, V, margin_line=0.0, margin_v=0.0):
        Vm = np.abs(V)
        ld = self.loading(V)
        P, Q = self.mismatch(lam, V)
        low = self.PQ & (Vm < self.vmin + margin_v)
        high = self.PQ & (Vm > self.vmax - margin_v)
        pq_vm = np.where(self.PQ, Vm, np.inf)
        return {"min_vm": float(pq_vm.min()), "weak_bus": int(np.argmin(pq_vm)),
                "n_low_v": int(low.sum()), "n_high_v": int(high.sum()), "n_over": int((ld >= 1.0 - margin_line).sum()),
                "max_loading": float(ld.max()) if len(ld) else np.nan,
                "resP_max_MW": float(P.max()), "resP_p99_MW": float(np.percentile(P, 99)), "resP_mean_MW": float(P.mean()),
                "resQ_max_Mvar": float(Q.max()) if len(Q) else np.nan,
                "mismatch_pu": float(max(P.max(), Q.max() if len(Q) else 0.0) / SN_MVA)}


# ---------------------------------------------------------------- the nose
def find_lambda_max(case, V_at_one=None, step=0.02, width=0.0025, lam_min=0.5, lam_cap=2.0, tol=1e-8, max_iter=30):
    """Largest load factor with an AC solution, by marching up from a solvable point and bisecting the first failure.
    At every failure the flat and DC starts are tried too, so a failure of the continuation start alone does not count.
    Returns a dict: lam_max, V_max (solution there), lam_first (first solvable point found), path [(lam, who)],
    n_solves, capped (True if solvable up to lam_cap)."""
    path, n = [], 0
    lam, V = 1.0, V_at_one
    if V is None:                       # the storm has no solution at lambda = 1: go down until one appears
        lam = 1.0 - step
        while lam >= lam_min:
            n += 3
            V, who = case.solve_any(lam, [("flat", case.flat), ("dc", case.dc), ("base", case.state(case.d["Vm"], case.d["Va"]))], max_iter, tol)
            if V is not None:
                path.append((lam, who)); break
            lam = round(lam - step, 6)
        if V is None:
            return {"lam_max": np.nan, "V_max": None, "lam_first": np.nan, "path": path, "n_solves": n, "capped": False}
    else:
        path.append((1.0, "storm"))
    lam_first = lam
    lo, V_lo = lam, V
    hi = None
    while hi is None:                   # march up
        trial = round(lo + step, 6)
        if trial > lam_cap:
            return {"lam_max": lo, "V_max": V_lo, "lam_first": lam_first, "path": path, "n_solves": n, "capped": True}
        n += 1
        V2, ok, _, _ = case.solve(trial, V_lo, max_iter, tol)
        if ok and np.all(np.isfinite(V2)) and np.abs(V2).min() > 0.2:
            lo, V_lo = trial, V2; path.append((trial, "continuation")); continue
        n += 2
        V2, who = case.solve_any(trial, [("flat", case.flat), ("dc", case.dc)], max_iter, tol)
        if V2 is not None:
            lo, V_lo = trial, V2; path.append((trial, who)); continue
        hi = trial
    while hi - lo > width:              # bisect
        mid = round(0.5 * (lo + hi), 8)
        n += 1
        V2, ok, _, _ = case.solve(mid, V_lo, max_iter, tol)
        if not (ok and np.all(np.isfinite(V2)) and np.abs(V2).min() > 0.2):
            n += 2
            V2, who = case.solve_any(mid, [("flat", case.flat), ("dc", case.dc)], max_iter, tol)
            ok = V2 is not None
        if ok:
            lo, V_lo = mid, V2; path.append((mid, "bisection"))
        else:
            hi = mid
    return {"lam_max": lo, "V_max": V_lo, "lam_first": lam_first, "path": path, "n_solves": n, "capped": False}


def truth_below_nose(case, lam_max, V_max, lams, tol=1e-8, max_iter=30):
    """AC solutions at the given load factors (all <= lam_max), marching down from the solution at the nose.
    Returns {lam: V} (missing where Newton failed, which should not happen below the nose)."""
    out, V = {}, V_max
    for lam in sorted(lams, reverse=True):
        V2, ok, _, _ = case.solve(lam, V, max_iter, tol)
        if not ok:
            V2, _ = case.solve_any(lam, [("flat", case.flat), ("dc", case.dc)], max_iter, tol)
            ok = V2 is not None
        if ok:
            out[lam] = V2; V = V2
    return out


def sigma_min(J, iters=40, seed=0):
    """Smallest singular value of the Jacobian, 1 / largest singular value of its inverse, by power iteration on
    J^-1 J^-T with one LU factorisation of J and one of J^T. Drops toward 0 at a genuine nose."""
    J = J.tocsc()
    lu, luT = splu(J), splu(J.T.tocsc())
    x = np.random.default_rng(seed).standard_normal(J.shape[0])
    x /= np.linalg.norm(x)
    val = np.nan
    for _ in range(iters):
        y = lu.solve(luT.solve(x))
        val = float(np.linalg.norm(y))
        x = y / val
    return 1.0 / np.sqrt(val)


# ---------------------------------------------------------------- helpers
def hop_distance(d, a, b):
    """Number of in-service lines between buses a and b (inf if not connected)."""
    nb, on = d["nb"], d["on"]
    A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
    return float(shortest_path(A, unweighted=True, directed=False, indices=int(a))[int(b)])


def auroc(score, positive):
    """Area under the ROC curve by ranks (ties get the average rank); nan if one class is empty."""
    score, positive = np.asarray(score, float), np.asarray(positive, bool)
    ok = np.isfinite(score); score, positive = score[ok], positive[ok]
    n1, n0 = positive.sum(), (~positive).sum()
    if n1 == 0 or n0 == 0:
        return np.nan
    order = np.argsort(score, kind="mergesort")
    ranks = np.empty(len(score)); ranks[order] = np.arange(1, len(score) + 1)
    s = score[order]
    i = 0
    while i < len(s):                   # average ranks of ties
        j = i
        while j + 1 < len(s) and s[j + 1] == s[i]:
            j += 1
        if j > i:
            ranks[order[i:j + 1]] = 0.5 * (i + j) + 1
        i = j + 1
    return float((ranks[positive].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


"""N core (reviewer items 4, 5, 9): generator reactive limits inside the AC truth and the check; stress patterns other
than the K1 scaling (loads only, reactive demand only, area-to-area transfer, generator outage on top of the storm);
warm-start timing of the classical solver. Uses LambdaCase / find_lambda_max / truth_below_nose from the K1 core and
build_ybus / newton / klu_solve / system_of / dc_angles / SN_MVA / HAVE_KLU from the step-1/6 core."""
import time

import numpy as np
import scipy.sparse as sp
from scipy.sparse.csgraph import dijkstra
from scipy.sparse.linalg import splu


# ================================================================ reactive limits
def bus_q_limits(d, gen_rows):
    """Per-bus reactive limits (p.u.) from the generator table of one scenario: sums over in-service generators.
    gen_rows: DataFrame with bus, min_q_mvar, max_q_mvar, in_service, is_slack_gen. Buses without a generator get 0/0."""
    nb = d["nb"]
    qmin, qmax = np.zeros(nb), np.zeros(nb)
    g = gen_rows[gen_rows["in_service"].to_numpy() > 0.5] if "in_service" in gen_rows else gen_rows
    np.add.at(qmin, g["bus"].to_numpy().astype(int), g["min_q_mvar"].to_numpy(float) / SN_MVA)
    np.add.at(qmax, g["bus"].to_numpy().astype(int), g["max_q_mvar"].to_numpy(float) / SN_MVA)
    return qmin, qmax


def q_generation(Y, V, Qd_pu):
    """Reactive generation implied by a voltage state (p.u.): injected Q plus the reactive load."""
    return (V * np.conj(Y @ V)).imag + Qd_pu


class QLimCase(LambdaCase):
    """A LambdaCase whose solver enforces generator reactive limits: a PV bus whose generation would exceed its limit
    is held at the limit as a PQ bus (and released again if its voltage recovers past the set-point). The outer loop
    is the standard one used in power-flow programs; a case counts as solved only when no further switching is needed."""

    def __init__(self, d, qmin_pu, qmax_pu, max_outer=15, eps=1e-6):
        super().__init__(d)
        self.qmin, self.qmax, self.max_outer, self.eps = qmin_pu, qmax_pu, max_outer, eps
        self.vset = d["Vm"].copy()                      # generator set-points (the base solution holds them)
        self.limited = self.PV & (self.qmax >= self.qmin)
        self.last = {"switched": {}, "outer": 0, "iters": 0}

    def _inner(self, S, V0, pv, pq, max_iter, tol):
        if HAVE_KLU:
            return klu_solve(self.Y, S, V0, self.ref, pv, pq, max_iter, tol)
        t0 = time.perf_counter()
        V, it, norms, _ = newton(self.Y, S, V0, pv, pq, tol=tol, max_it=max_iter)
        return V, bool(norms[-1] <= tol), it, time.perf_counter() - t0

    def solve(self, lam, V0, max_iter=30, tol=1e-8):
        """Newton-Raphson with reactive-limit switching. Returns V, converged, total inner iterations, seconds.
        `self.last` keeps the switched buses of the last call ({bus: 'max' | 'min'}) and the number of outer rounds."""
        t0 = time.perf_counter()
        S_base = self.S(lam)
        Qd = lam * self.Qd
        V = np.asarray(V0, complex).copy()
        switched, backs, total_it = {}, {}, 0
        for outer in range(1, self.max_outer + 1):
            sw = np.array(sorted(switched), dtype=int)
            pv = np.array([b for b in self.pv if b not in switched], dtype=int)
            pq = np.array(sorted(set(self.pq.tolist()) | set(switched)), dtype=int)
            S = S_base.copy()
            if len(sw):
                S[sw] = S_base.real[sw] + 1j * (np.array([self.qmax[b] if switched[b] == "max" else self.qmin[b] for b in sw]) - Qd[sw])
            V[pv] = self.vset[pv] * np.exp(1j * np.angle(V[pv]))      # PV buses start at their set-point
            V, ok, it, _ = self._inner(S, V, pv, pq, max_iter, tol)
            total_it += it
            if not (ok and np.all(np.isfinite(V)) and np.abs(V).min() > 0.2):
                self.last = {"switched": dict(switched), "outer": outer, "iters": total_it}
                return V, False, total_it, time.perf_counter() - t0
            Qg = q_generation(self.Y, V, Qd)
            to_max = [b for b in pv if self.limited[b] and Qg[b] > self.qmax[b] + self.eps]
            to_min = [b for b in pv if self.limited[b] and Qg[b] < self.qmin[b] - self.eps]
            Vm = np.abs(V)
            back = [b for b in sw if backs.get(b, 0) < 2 and ((switched[b] == "max" and Vm[b] > self.vset[b] + self.eps) or
                                                             (switched[b] == "min" and Vm[b] < self.vset[b] - self.eps))]
            if not to_max and not to_min and not back:
                self.last = {"switched": dict(switched), "outer": outer, "iters": total_it}
                return V, True, total_it, time.perf_counter() - t0
            for b in to_max:
                switched[int(b)] = "max"
            for b in to_min:
                switched[int(b)] = "min"
            for b in back:
                del switched[int(b)]; backs[int(b)] = backs.get(int(b), 0) + 1
        self.last = {"switched": dict(switched), "outer": self.max_outer, "iters": total_it}
        return V, False, total_it, time.perf_counter() - t0

    def decisions(self, lam, V, margin_line=0.0, margin_v=0.0):
        """As LambdaCase.decisions, but voltage limits are checked at every non-slack bus (a generator at its limit no
        longer holds its voltage) and the reactive generation beyond the limits is reported."""
        Vm = np.abs(V)
        ld = self.loading(V)
        P, Q = self.mismatch(lam, V)
        chk = ~self.REF
        low = chk & (Vm < self.vmin + margin_v)
        high = chk & (Vm > self.vmax - margin_v)
        vm_chk = np.where(chk, Vm, np.inf)
        Qg = q_generation(self.Y, V, lam * self.Qd)
        over_q = self.limited & (Qg > self.qmax + 1e-6)
        return {"min_vm": float(vm_chk.min()), "weak_bus": int(np.argmin(vm_chk)),
                "n_low_v": int(low.sum()), "n_high_v": int(high.sum()), "n_over": int((ld >= 1.0 - margin_line).sum()),
                "max_loading": float(ld.max()) if len(ld) else np.nan,
                "resP_max_MW": float(P.max()), "resP_p99_MW": float(np.percentile(P, 99)), "resP_mean_MW": float(P.mean()),
                "resQ_max_Mvar": float(Q.max()) if len(Q) else np.nan,
                "mismatch_pu": float(max(P.max(), Q.max() if len(Q) else 0.0) / SN_MVA),
                "n_gen_over_qmax": int(over_q.sum()), "q_over_qmax_Mvar": float(SN_MVA * np.clip(Qg - self.qmax, 0, None)[self.limited].sum())}


# ================================================================ stress patterns
class StressCase(LambdaCase):
    """A storm topology stressed in one direction by a factor lambda (1 = the actual operating point):
    'all'      every load and every non-slack generator scaled together (the K1 pattern; slack takes the rest);
    'load'     loads only (P and Q at constant power factor); generation fixed, the slack takes the growth;
    'reactive' reactive demand only (Q of every load); active power unchanged;
    'transfer' loads inside an area scaled, generation outside the area scaled up by the same MW (a transfer into
               the storm area); the slack takes the losses."""

    def __init__(self, d, mode="all", area=None, kappa=None):
        super().__init__(d)
        self.mode = mode
        self.Pg, self.Pd = d["Pg"] / SN_MVA, d["Pd"] / SN_MVA
        self.area = np.zeros(self.nb, bool) if area is None else np.asarray(area, bool)
        self.genB = (self.Pg > 0) & ~self.area & ~self.REF
        self.kappa = kappa if kappa is not None else (self.Pd[self.area].sum() / max(self.Pg[self.genB].sum(), 1e-9))

    def S(self, lam):
        if self.mode == "all":
            return lam * self.P0 + 1j * (self.Qg - lam * self.Qd)
        if self.mode == "load":
            return (self.Pg - lam * self.Pd) + 1j * (self.Qg - lam * self.Qd)
        if self.mode == "reactive":
            return self.P0 + 1j * (self.Qg - lam * self.Qd)
        if self.mode == "transfer":
            fl = 1.0 + (lam - 1.0) * self.area
            fg = 1.0 + (lam - 1.0) * self.kappa * self.genB
            return (self.Pg * fg - self.Pd * fl) + 1j * (self.Qg - self.Qd * fl)
        raise ValueError(self.mode)

    def dc_loading(self, lam):
        """DC loading at this lambda: angles from the scaled active injections through the DC susceptance matrix."""
        d = self.d
        nb, on = d["nb"], d["on"]
        f, t, b = d["f"][on], d["t"][on], d["Yft"][on].imag
        B = sp.csr_matrix((np.r_[b, b, -b, -b], (np.r_[f, t, f, t], np.r_[f, t, t, f])), shape=(nb, nb)).tocsc()
        keep = self.nonref
        th = np.zeros(nb)
        th[keep] = splu(B[keep][:, keep]).solve(self.S(lam).real[keep])
        return self.loading(np.exp(1j * (th + np.radians(float(d["Va"][self.ref[0]])))))


def storm_area(d, footprint_buses, min_load_share=0.10, max_hops=6):
    """Buses within h hops of the storm footprint, h grown until the area holds at least `min_load_share` of the load."""
    nb, on = d["nb"], d["on"]
    A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
    hop = dijkstra(A, directed=False, unweighted=True, indices=np.asarray(footprint_buses, int), min_only=True)
    total = max(d["Pd"].sum(), 1e-9)
    for h in range(0, max_hops + 1):
        area = hop <= h
        if d["Pd"][area].sum() >= min_load_share * total:
            return area, h, hop
    return hop <= max_hops, max_hops, hop


def footprint_of(d, removed_rows):
    """Endpoint buses of the removed lines (branch-row indices)."""
    return np.unique(np.r_[d["f"][removed_rows], d["t"][removed_rows]])


def pick_generator_outage(d, gen_rows, hop, max_hops=3):
    """The largest in-service non-slack generator (by active output; by reactive range if none produces active power)
    whose bus lies within `max_hops` of the storm footprint; the largest anywhere if there is none. Returns its row
    label in gen_rows (or None)."""
    g = gen_rows
    keep = np.ones(len(g), bool)
    if "in_service" in g:
        keep &= g["in_service"].to_numpy(float) > 0.5
    if "is_slack_gen" in g:
        keep &= g["is_slack_gen"].to_numpy(float) < 0.5
    keep &= ~d["REF"][g["bus"].to_numpy().astype(int)]
    g = g[keep]
    if len(g) == 0:
        return None
    size = g["p_mw"].to_numpy(float)
    if not (size > 0).any():
        size = g["max_q_mvar"].to_numpy(float) - g["min_q_mvar"].to_numpy(float)
    near = hop[g["bus"].to_numpy().astype(int)] <= max_hops
    cand = g[near] if near.any() else g
    size_c = size[near] if near.any() else size
    return int(cand.index[int(np.argmax(size_c))])


def remove_generator(d, gen_rows, row):
    """The truth model after one generator is lost: its P and Q leave the bus (the slack takes the P); if no in-service
    generator remains at the bus, it becomes a PQ bus. Returns the new dict and the reduced generator table."""
    d2 = {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in d.items()}
    g = gen_rows.copy()
    bus = int(g.loc[row, "bus"])
    d2["Pg"][bus] -= float(g.loc[row, "p_mw"]); d2["Qg"][bus] -= float(g.loc[row, "q_mvar"])
    g = g.drop(index=row)
    left = g[(g["bus"].to_numpy().astype(int) == bus) & (g["in_service"].to_numpy() > 0.5)] if "in_service" in g else g[g["bus"] == bus]
    if len(left) == 0 and not d2["REF"][bus]:
        d2["PV"][bus] = False; d2["PQ"][bus] = True
        d2["Pg"][bus] = 0.0; d2["Qg"][bus] = 0.0
    return d2, g, bus


# ================================================================ warm starts (item 9)
def timed_solves(case, lam, starts, max_iter=30, tol=1e-8, reps=3):
    """Iterations and the median time (ms) of the solver from each start; converged flag per start."""
    out = {}
    for name, V0 in starts:
        its, ts, oks = [], [], []
        for _ in range(reps):
            V, ok, it, dt = case.solve(lam, V0, max_iter, tol)
            its.append(it); ts.append(1000 * dt); oks.append(bool(ok and np.all(np.isfinite(V)) and np.abs(V).min() > 0.2))
        out[name] = {"converged": all(oks), "iters": int(np.median(its)), "ms": float(np.median(ts))}
    return out


# ---------------------------------------------------------------- notebook setup
import json, math, platform, zipfile, warnings, copy
import matplotlib.pyplot as plt
from IPython.display import display
from gridfm_graphkit.datasets.globals import PD_H, QD_H, PG_H

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:.4g}")
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(SEED)

# N-2 scenarios that are hard for GENCO: not used as storm base points, so the base points are ordinary
HARD2 = set(json.loads('[21, 22, 129, 131, 133, 158, 171, 436, 552, 555, 1231, 1251, 1264, 1282, 1294, 1396, 1446, 1449, 1457, 1521, 1574, 1578, 2212, 2222, 2288, 2353, 2390, 2394, 3314, 3379, 3851, 3852, 3900, 3903, 3905, 3927, 3928, 3947, 4622, 4628, 4698, 4711, 4722, 4723, 4762, 4777, 4778, 4779, 4798, 4801, 4851, 4853, 4878, 4896, 4898, 4899, 4902, 4920, 4923, 4924, 4961, 5231, 5258, 5368, 5382, 5949, 6025, 6057, 6208, 6230, 6294, 6319, 6320, 6356, 6367, 7259, 7261, 7356, 7658, 7684, 7689, 7844, 7863, 7870, 7882, 7968, 8188, 8255, 8411, 8487, 8595]'))
RAW3_PAT = re.compile(r"(?:^|/)(bus_data|branch_data|gen_data)\.parquet/scenario_partition=(\d+)/[^/]+$")
PART = 200


def read_all(path, scen):
    dset = ds.dataset(str(path), format="parquet", partitioning="hive")
    values = pa.array(np.asarray(scen, dtype=np.int64)).cast(dset.schema.field("scenario").type)
    t = dset.to_table(filter=ds.field("scenario").isin(values)).to_pandas()
    return t.drop(columns=[c for c in ["scenario_partition"] if c in t.columns])


def pick_storm(d, extra, rng):
    """`extra` more lines to remove, the nearest (in hops) to a random centre bus, keeping the grid in one piece.
    Returns branch-row indices, the storm radius in hops and the centre bus; (None, None, centre) if impossible."""
    nb = d["nb"]; on = d["on"].copy()
    cand = np.nonzero(on & (d["rate"] > 0))[0]
    center = int(rng.integers(nb))
    A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
    hop = dijkstra(A, directed=False, unweighted=True, indices=center)
    dist = np.minimum(hop[d["f"][cand]], hop[d["t"][cand]])
    order = cand[np.lexsort((rng.random(len(cand)), dist))]
    removed = []
    for i in order:
        if len(removed) == extra:
            break
        trial = on.copy(); trial[i] = False
        if connected(nb, d["f"][trial], d["t"][trial]):
            on = trial; removed.append(int(i))
    if len(removed) < extra:
        return None, None, center
    radius = float(np.max(np.minimum(hop[d["f"][removed]], hop[d["t"][removed]])))
    return np.array(removed), radius, center


print("ready | lightsim2grid (KLU) available:", HAVE_KLU)


def truth_dicts(level, ids):
    """Stored AC solutions of the given scenarios as compact dicts (the truth sits in the prediction slot too)."""
    ids = np.sort(np.asarray(ids, dtype=np.int64))
    bus = read_cols(find_local(LOCAL_DIR, level, "bus_data"), ["scenario", "bus", "Vm", "Va"], scen=ids)
    pred = pd.DataFrame({"scenario": bus.scenario.astype(np.int64), "bus": bus.bus.astype(np.int64),
                         "vm": bus.Vm, "va": np.radians(bus.Va)}).sort_values(["scenario", "bus"]).reset_index(drop=True)
    A = load_chunk(find_local(LOCAL_DIR, level, "bus_data"), find_local(LOCAL_DIR, level, "branch_data"), pred, ids)
    return {int(ids[i]): extract_scenario(A, i) for i in range(len(ids))}

## 1. The cases of the frozen evaluation, from their stored lists; download of the model and the two data blocks

In [ ]:
# The 870 cases of the frozen final evaluation, rebuilt from their stored base points and removed-line lists (not re-drawn)
import base64, io
F1_CASES = pd.read_csv(io.StringIO(base64.b64decode(F1_CASES_CSV_B64).decode()))
F1_WF = pd.read_csv(io.StringIO(base64.b64decode(F1_WORKFLOWS_CSV_B64).decode()))
F1_CASES["removed_lines"] = F1_CASES["removed_lines"].map(json.loads)
BASE_IDS = np.array(sorted(F1_CASES["base"].unique()), dtype=np.int64)
print(f"{len(F1_CASES)} F1 cases on {len(BASE_IDS)} base points; families: {F1_CASES.family.value_counts().to_dict()}")
print(f"F1 workflow rows for the comparison: {len(F1_WF)} ({F1_WF.workflow.unique().tolist()})")

In [ ]:
from huggingface_hub import HfApi, snapshot_download
print("Listing the repository ...")
all_files = [(f.path, f.size) for f in HfApi().list_repo_tree(REPO_ID, recursive=True, revision=HF_REVISION) if getattr(f, "size", None) is not None]
local_has = lambda p: (Path(LOCAL_DIR) / p).exists()
plan = {p: s for p, s in all_files if p.startswith(MODEL_DIR + "/")}
for p, s in all_files:
    m = RAW3_PAT.search(p) if p.startswith(f"data/test_{LEVEL_BASE}/") else None
    if m and int(m.group(2)) in BASE_PARTS:
        plan[p] = s
new = sum(s for p, s in plan.items() if not local_has(p)) / 1e9
print(f"Planned: {len(plan)} files, {new:.2f} GB not yet on disk")
if new > MAX_DOWNLOAD_GB:
    raise RuntimeError("Planned download is above MAX_DOWNLOAD_GB")
snapshot_download(REPO_ID, allow_patterns=list(plan), local_dir=LOCAL_DIR, revision=HF_REVISION)
print("Download finished")

In [ ]:
BASE = truth_dicts(LEVEL_BASE, BASE_IDS)
missing = [int(b) for b in BASE_IDS if int(b) not in BASE]
if missing:
    raise RuntimeError(f"{len(missing)} base points of F1 are not on disk (first: {missing[:5]}); the data blocks {BASE_PARTS} must be the ones F1 used")
NB = next(iter(BASE.values()))["nb"]
CASES = {}
for r_ in F1_CASES.itertuples(index=False):
    d = BASE[int(r_.base)]
    rows = np.nonzero(np.isin(d["line"], np.asarray(r_.removed_lines, dtype=np.int64)))[0]
    if len(rows) != len(r_.removed_lines):
        raise RuntimeError(f"case {r_.sid}: {len(r_.removed_lines)} removed lines in F1's list, {len(rows)} found in the base point's table")
    d2 = {kk: (v.copy() if isinstance(v, np.ndarray) else v) for kk, v in d.items()}
    d2["on"][rows] = False
    d2.pop("Va_dc", None)
    CASES[int(r_.sid)] = {"sid": int(r_.sid), "base": int(r_.base), "family": r_.family, "removed_lines": list(r_.removed_lines), "rows": rows, "d": d2}
print(f"{len(CASES)} cases rebuilt exactly from F1's lists on {len(BASE)} operating points ({NB} buses)")

## 2. GENCO, and a check of the model

In [ ]:
# [GPU] build graphs of the base points (used for the model check)

import yaml
cfg_path = os.path.join(GRAPHKIT_DIR, "scripts", "contingency", "configs", "caseTexas_eval.yaml")
with open(cfg_path) as f:
    ARGS = NestedNamespace(**yaml.safe_load(f))
stats = torch.load(os.path.join(LOCAL_DIR, MODEL_DIR, "normalizer_stats.pt"), map_location="cpu", weights_only=True)
stats_net = stats[NETWORK] if NETWORK in stats else (stats if "baseMVA" in stats else next(iter(stats.values())))
NORMALIZER = load_normalizer(ARGS)
NORMALIZER.fit_from_dict(stats_net)
TRANSFORM = get_task_transforms(ARGS)
print("normalizer baseMVA:", NORMALIZER.baseMVA, "| config:", cfg_path)

BUS_T = read_all(find_local(LOCAL_DIR, LEVEL_BASE, "bus_data"), BASE_IDS)
GEN_T = read_all(find_local(LOCAL_DIR, LEVEL_BASE, "gen_data"), BASE_IDS)
BR_T = read_all(find_local(LOCAL_DIR, LEVEL_BASE, "branch_data"), BASE_IDS)
BUS_T = merge_agg_gen_into_bus(BUS_T, GEN_T)
BG, GG, RG = BUS_T.groupby("scenario"), GEN_T.groupby("scenario"), BR_T.groupby("scenario")


def build_graph(sid, base, Pd=None, Qd=None, removed_lines=()):
    """GENCO's input graph for base point `base` with extra lines removed and (optionally) other loads."""
    b = BG.get_group(base).sort_values("bus").reset_index(drop=True).copy()
    if Pd is not None:
        b["Pd"], b["Qd"] = np.asarray(Pd, float), np.asarray(Qd, float)
    r = RG.get_group(base).copy()
    if len(removed_lines):
        r.loc[r["idx"].astype(int).isin([int(x) for x in removed_lines]), "br_status"] = 0.0
    g = build_hetero_data_for_scenario(int(sid), b, GG.get_group(base), r)
    NORMALIZER.transform(g)
    return TRANSFORM(g)


def scaled_graph(g, lam):
    """The same graph with every load and every non-slack generator scaled by lam (the slack's generation is a
    masked input, 0, and stays 0; loads and generation are divided by baseMVA, so scaling the inputs is exact)."""
    h = g.clone()
    h["bus"].x = h["bus"].x.clone(); h["gen"].x = h["gen"].x.clone()
    h["bus"].x[:, PD_H] *= lam; h["bus"].x[:, QD_H] *= lam; h["gen"].x[:, PG_H] *= lam
    return h


BASE_GRAPHS = [build_graph(int(s), int(s)) for s in BASE_IDS[:N_CHECK]]
print(f"{len(BASE_GRAPHS)} base graphs built")

In [ ]:
# [GPU] load model

TASK = get_task(ARGS, [NORMALIZER])
sd = torch.load(os.path.join(LOCAL_DIR, MODEL_DIR, "best_model_state_dict.pt"), map_location="cpu")
sd = adapt_state_dict_for_model(sd, TASK)
missing_keys, unexpected_keys = TASK.load_state_dict(sd, strict=False)
print(f"checkpoint loaded: {len(sd)} tensors | missing {len(missing_keys)} | unexpected {len(unexpected_keys)}")
if missing_keys:
    raise RuntimeError(f"The checkpoint does not match the model. Missing: {missing_keys[:5]}")
MODEL = TASK.model.to(DEVICE).eval()
print("GENCO parameters:", sum(p.numel() for p in MODEL.parameters()))


def forward(model, batch):
    return model(batch.x_dict, batch.edge_index_dict, batch.edge_attr_dict, batch.mask_dict)

In [ ]:
# [GPU] run GENCO on a list of graphs; check the model against the stored AC solutions of the base points

def run_genco(graphs, bs=BATCH):
    """GENCO's (Vm, Va in radians) for each graph, in order."""
    out = [None] * len(graphs)
    for i, h in enumerate(graphs):
        h["scenario_id"] = torch.tensor([i], dtype=torch.long)
    for b in DataLoader(graphs, batch_size=bs, shuffle=False):
        b = b.to(DEVICE)
        with torch.no_grad():
            o = forward(MODEL, b)["bus"].float()
        ptr = b["bus"].ptr.cpu().tolist()
        for g, i in enumerate(b["scenario_id"].view(-1).cpu().tolist()):
            out[int(i)] = (o[ptr[g]:ptr[g + 1], VM_OUT].cpu().numpy().astype(np.float64),
                           o[ptr[g]:ptr[g + 1], VA_OUT].cpu().numpy().astype(np.float64))
    return out


t0 = time.perf_counter()
BASE_OUT = run_genco(BASE_GRAPHS[:N_CHECK])
T_FWD_MS = 1000 * (time.perf_counter() - t0) / len(BASE_OUT)
errs = np.array([np.max(np.abs(vm - BASE[int(s)]["Vm"])[BASE[int(s)]["PQ"]]) for s, (vm, va) in zip(BASE_IDS[:N_CHECK], BASE_OUT)])
print(f"GENCO on {len(BASE_OUT)} base points: worst-bus voltage error median {np.median(errs):.4f} p.u., "
      f"95th percentile {np.percentile(errs, 95):.4f} p.u. (expected about 0.001 and 0.004) | {T_FWD_MS:.0f} ms per graph incl. loading")
if np.median(errs) > 0.02 and NB > 100:
    raise RuntimeError("GENCO's error on ordinary N-2 cases is far above the known level: the model did not load correctly; check the installation and the downloaded checkpoint.")

In [ ]:
def case_graph(c):
    return build_graph(c["sid"], c["base"], removed_lines=c["removed_lines"])


def genco_one(c):
    """GENCO's state for one case (the released model as it is: float32, eager, batch 1)."""
    g = case_graph(c)
    g["scenario_id"] = torch.tensor([0], dtype=torch.long)
    b = next(iter(DataLoader([g], batch_size=1, shuffle=False))).to(DEVICE)
    with torch.no_grad():
        o = forward(MODEL, b)["bus"].float()
    vm = o[:, VM_OUT].cpu().numpy().astype(np.float64); va = np.degrees(o[:, VA_OUT].cpu().numpy().astype(np.float64))
    return vm, va

## 3. Every start on every case: the frozen outcome first, the other solutions next to it

In [ ]:
# Every case through the two workflows with the branch check on: the first converged start is the frozen workflow's outcome
# (it must reproduce frozen-test's status and start), and every further start is run and compared with it.
POL = dict(POLICY, **BRANCH)
WF = {"genco_verify": ("genco", "base", "dc", "flat"), "classical": ("base", "dc", "flat")}
F1_REF = {(int(r_.sid), r_.workflow): r_ for r_ in F1_WF.itertuples(index=False)}
rows, SOL = [], {}
t_all = time.perf_counter()
for i, sid in enumerate(sorted(CASES)):
    c = CASES[sid]
    vm, va = genco_one(c)
    vf = Verifier(c["d"], POL)
    for w, order in WF.items():
        r = vf.run(genco_vm=vm, genco_va_deg=va, policy_name=w, order=order) if w == "genco_verify" else vf.run(policy_name=w, order=order)
        f1 = F1_REF.get((sid, w))
        row = {"sid": sid, "base": c["base"], "family": c["family"], "workflow": w, "status": r.status, "start_used": r.start_used, "iterations": r.iterations,
               "min_vm_all": r.limits.get("min_vm_all_buses"), "n_over": r.limits.get("n_over"), "n_low_v": r.limits.get("n_low_v"), "n_high_v": r.limits.get("n_high_v"),
               "low_voltage_flag": r.flags.get("low_voltage_branch_suspect", False),
               "n_starts_converged": r.flags.get("n_starts_converged", 0), "n_solutions_found": r.flags.get("n_solutions_found", 0),
               "multiple_solutions": r.flags.get("multiple_solutions", False), "decisions_differ": r.flags.get("decisions_differ", False),
               "f1_status": f1.status if f1 is not None else None, "f1_start_used": f1.start_used if f1 is not None else None,
               "f1_min_vm_all": f1.min_vm_all if f1 is not None else np.nan,
               "same_status_as_f1": (f1 is not None and f1.status == r.status),
               "same_start_as_f1": (f1 is not None and (str(f1.start_used) if isinstance(f1.start_used, str) else "none") == (r.start_used or "none")),
               "solutions": json.dumps(r.flags.get("solutions", []), default=float),
               "highest_voltage_start": None, "highest_voltage_min_vm": np.nan, "highest_voltage_violation": None, "highest_voltage_same_decisions": None}
        if r.converged and r.flags.get("multiple_solutions"):
            hi = max(r.flags["solutions"], key=lambda s_: s_["min_vm_all_buses"])
            row.update({"highest_voltage_start": hi["start"], "highest_voltage_min_vm": hi["min_vm_all_buses"], "highest_voltage_violation": hi["violation"],
                        "highest_voltage_same_decisions": hi["same_decisions_as_chosen"]})
        rows.append(row)
    if (i + 1) % 100 == 0:
        print(f"  {i + 1} cases ({time.perf_counter() - t_all:.0f} s)")
B = pd.DataFrame(rows)
B.to_csv(Path(OUT_DIR) / "runs.csv", index=False)
print(f"{len(CASES)} cases x {len(WF)} workflows in {time.perf_counter() - t_all:.0f} s")

## 4. Consistency with the frozen run, and the ambiguity the branch check finds

In [ ]:
# 1. Consistency with the frozen run: the first converged start must give frozen-test's status and start on every case.
cons = B.groupby("workflow").agg(cases=("sid", "count"), same_status=("same_status_as_f1", "sum"), same_start=("same_start_as_f1", "sum"))
display(cons)
bad = B[~B.same_status_as_f1 | ~B.same_start_as_f1]
if len(bad):
    print(f"{len(bad)} rows differ from F1 (status or start):"); display(bad[["sid", "family", "workflow", "status", "f1_status", "start_used", "f1_start_used", "min_vm_all", "f1_min_vm_all"]].head(30))
else:
    print("every status and every start reproduce the frozen run: the revised module changes nothing on the frozen path")
# 2. The branch check: how many cases have more than one solution among the starts, and do the decisions differ?
ver = B[B.status != "unresolved"]
amb = ver.groupby(["workflow", "family"]).agg(verified=("sid", "count"), starts_converged=("n_starts_converged", "mean"),
                                             multiple_solutions=("multiple_solutions", "sum"), decisions_differ=("decisions_differ", "sum"),
                                             low_voltage_flag=("low_voltage_flag", "sum"))
display(amb)
tot = ver.groupby("workflow").agg(verified=("sid", "count"), multiple_solutions=("multiple_solutions", "sum"), decisions_differ=("decisions_differ", "sum"))
display(tot)
multi = ver[ver.multiple_solutions].copy()
if len(multi):
    print(f"{multi.sid.nunique()} cases with more than one solution among the starts of at least one workflow:")
    display(multi[["sid", "family", "workflow", "start_used", "min_vm_all", "n_solutions_found", "decisions_differ", "highest_voltage_start", "highest_voltage_min_vm", "highest_voltage_same_decisions"]].sort_values(["sid", "workflow"]).head(40))
# 3. The two cases found after the frozen run
for sid in KNOWN_CASES:
    sub = B[B.sid == sid]
    if len(sub):
        print(f"case {sid}:"); display(sub[["workflow", "status", "start_used", "min_vm_all", "n_solutions_found", "multiple_solutions", "decisions_differ", "highest_voltage_start", "highest_voltage_min_vm"]])
# 4. What the highest-voltage rule would change
hv = ver[ver.multiple_solutions]
changed = hv[(hv.highest_voltage_start != hv.start_used) & ~hv.highest_voltage_same_decisions.map(lambda v: True if v is None or (isinstance(v, float) and np.isnan(v)) else bool(v))]
print(f"cases where the highest-voltage rule picks another solution with other decisions than the first start: {len(changed)} rows ({changed.sid.nunique()} cases)")
SUMMARY_B1 = {"cases": int(len(CASES)), "consistency_with_f1": cons.to_dict(orient="index"), "rows_differing_from_f1": int(len(bad)),
              "by_workflow": tot.to_dict(orient="index"), "by_workflow_family": {f"{k[0]}|{k[1]}": v for k, v in amb.to_dict(orient="index").items()},
              "cases_with_multiple_solutions": sorted(int(x) for x in multi.sid.unique()),
              "cases_where_highest_voltage_rule_changes_decisions": sorted(int(x) for x in changed.sid.unique()),
              "known_cases": {int(s): B[B.sid == s][["workflow", "status", "start_used", "min_vm_all", "n_solutions_found", "multiple_solutions", "decisions_differ"]].to_dict(orient="records") for s in KNOWN_CASES if (B.sid == s).any()}}

In [ ]:
import matplotlib.pyplot as plt
fams = sorted(ver.family.unique())
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for ax, w in zip(axes, WF):
    g = ver[ver.workflow == w].groupby("family").agg(n=("sid", "count"), m=("multiple_solutions", "sum"), d=("decisions_differ", "sum")).reindex(fams).fillna(0)
    x = np.arange(len(fams))
    ax.bar(x - 0.2, g.n, 0.4, color="#dddddd", label="verified cases")
    ax.bar(x + 0.2, g.m, 0.4, color="#2a78d6", label="more than one solution among the starts")
    ax.bar(x + 0.2, g.d, 0.4, color="#c8102e", label="... with different decisions")
    ax.set_xticks(x); ax.set_xticklabels(fams, rotation=20); ax.set_title(f"{w}: starts {' → '.join(WF[w])}", fontsize=9)
    ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(Path(OUT_DIR) / "fig_branch_check.png", dpi=150); plt.show()

## 5. Summary and output files

In [ ]:
import platform, shutil
SUMMARY_B1.update({"method": "revised method, not the frozen test: the F1 cases rebuilt from their stored lists, run with branch_check on "
                             "(every start run, converged states grouped into distinct solutions, decisions compared); the first converged "
                             "start is the frozen workflow's outcome and is compared with F1's record",
                   "policy": POL, "workflows": WF, "data_partitions": BASE_PARTS,
                   "graphkit": {"checked_out_commit": subprocess.run(["git", "-C", GRAPHKIT_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()},
                   "versions": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "torch": torch.__version__,
                                "device": str(DEVICE), "lightsim2grid": HAVE_KLU}})
json.dump(SUMMARY_B1, open(Path(OUT_DIR) / "summary.json", "w"), indent=1, default=str)
print("VERDICT")
print(f"- Frozen path reproduced: {int(B.same_status_as_f1.sum())} of {len(B)} rows with F1's status, {int(B.same_start_as_f1.sum())} with F1's start.")
for w in WF:
    t_ = tot.loc[w]
    print(f"- {w}: {int(t_.multiple_solutions)} of {int(t_.verified)} verified cases have more than one solution among the starts; in {int(t_.decisions_differ)} the decisions differ.")
print(f"- The highest-voltage rule would change the decisions of {changed.sid.nunique()} cases.")
shutil.make_archive(OUT_DIR, "zip", OUT_DIR)
print(f"written {OUT_DIR}.zip")